In [ ]:
!pip -q install awscli

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

os.environ["AWS_ACCESS_KEY_ID"] = secrets.get_secret("R2_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = secrets.get_secret("R2_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"] = "auto"

account_id = secrets.get_secret("R2_ACCOUNT_ID")
bucket = secrets.get_secret("R2_BUCKET")

os.environ["R2_ENDPOINT"] = f"https://{account_id}.r2.cloudflarestorage.com"
os.environ["R2_BUCKET"] = bucket

print("R2 endpoint:", os.environ["R2_ENDPOINT"])
print("R2 bucket:", os.environ["R2_BUCKET"])

In [ ]:
!aws --endpoint-url "$R2_ENDPOINT" s3 ls "s3://$R2_BUCKET"

In [ ]:
from pathlib import Path
import re
import subprocess
import pandas as pd
from collections import defaultdict

DATASET_ROOTS = [
    Path("/kaggle/input/datasets/loinguyen57/aic-dataset-2025-s1"),
    Path("/kaggle/input/datasets/loinguyen57/aic-dataset-2025-s2"),
    Path("/kaggle/input/datasets/loinguyen57/aic-dataset-s2-video"),
]

EXCLUDE_VIDEO_FOLDERS = {
    "Videos_L25_a1",
    "Videos_L25_b",
}

for root in DATASET_ROOTS:
    print(root, "exists:", root.exists())

In [ ]:
def get_keyframe_group(folder_name: str):
    """
    Keyframes_L21   -> L21
    Keyframes_L26_a -> L26
    Keyframes_K01   -> K01
    """
    m = re.match(r"Keyframes_([LK]\d+)", folder_name)
    return m.group(1) if m else None


def get_video_group(folder_name: str):
    """
    Videos_L21_a  -> L21
    Videos_L25_a  -> L25
    Videos_L25_a1 -> L25
    Videos_K01    -> K01
    """
    m = re.match(r"Videos_([LK]\d+)", folder_name)
    return m.group(1) if m else None


def r2_sync(src: Path, dst_prefix: str, dry_run: bool = False):
    cmd = [
        "aws",
        "--endpoint-url", os.environ["R2_ENDPOINT"],
        "s3", "sync",
        str(src),
        f"s3://{os.environ['R2_BUCKET']}/{dst_prefix}",
        "--size-only",
        "--no-progress",
        "--only-show-errors",
        "--cache-control", "public, max-age=31536000, immutable",
    ]

    if dry_run:
        cmd.append("--dryrun")

    print("\nSYNC:")
    print("  FROM:", src)
    print("  TO:  ", f"s3://{os.environ['R2_BUCKET']}/{dst_prefix}")
    subprocess.run(cmd, check=True)

In [ ]:
keyframe_folders = []
video_folders = []
skipped_video_folders = []

for dataset_root in DATASET_ROOTS:
    for folder in dataset_root.iterdir():
        if not folder.is_dir():
            continue

        kf_group = get_keyframe_group(folder.name)
        if kf_group and (folder / "keyframes").exists():
            keyframe_folders.append((folder, kf_group))

        vd_group = get_video_group(folder.name)
        if vd_group and (folder / "video").exists():
            if folder.name in EXCLUDE_VIDEO_FOLDERS:
                skipped_video_folders.append((folder, vd_group))
            else:
                video_folders.append((folder, vd_group))

print("Keyframe folders:", len(keyframe_folders))
for f, g in keyframe_folders:
    print(" ", g, f)

print("\nVideo folders to upload:", len(video_folders))
for f, g in video_folders:
    print(" ", g, f)

print("\nVideo folders skipped:", len(skipped_video_folders))
for f, g in skipped_video_folders:
    print(" ", g, f)

In [ ]:
video_dest_map = defaultdict(list)

for folder, group in video_folders:
    video_dir = folder / "video"

    for mp4 in video_dir.glob("*.mp4"):
        dest_key = f"Videos/Videos_{group}/{mp4.name}"
        video_dest_map[dest_key].append(str(mp4))

collisions = {k: v for k, v in video_dest_map.items() if len(v) > 1}

print("Video destination collisions:", len(collisions))

for k, v in list(collisions.items())[:50]:
    print("\nCOLLISION:", k)
    for src in v:
        print("  ", src)

In [ ]:
# for folder, group in keyframe_folders:
#     src = folder / "keyframes"
#     dst = f"Keyframes/Keyframes_{group}/"
#     r2_sync(src, dst, dry_run=True)

# for folder, group in video_folders:
#     src = folder / "video"
#     dst = f"Videos/Videos_{group}/"
#     r2_sync(src, dst, dry_run=True)

In [ ]:
for folder, group in keyframe_folders:
    src = folder / "keyframes"
    dst = f"Keyframes/Keyframes_{group}/"
    r2_sync(src, dst, dry_run=False)

In [ ]:
# for folder, group in video_folders:
#     src = folder / "video"
#     dst = f"Videos/Videos_{group}/"
#     r2_sync(src, dst, dry_run=False)

In [ ]:
rows = []

# Keyframes
for folder, group in keyframe_folders:
    src = folder / "keyframes"

    for img in src.glob("*/*.jpg"):
        video_id = img.parent.name
        frame_name = img.name
        frame_stem = img.stem
        frame_idx = int(frame_stem) if frame_stem.isdigit() else None

        rows.append({
            "kind": "keyframe",
            "group": group,
            "video_id": video_id,
            "frame_name": frame_name,
            "frame_idx": frame_idx,
            "source_path": str(img),
            "r2_key": f"Keyframes/Keyframes_{group}/{video_id}/{frame_name}",
        })

# Videos
for folder, group in video_folders:
    src = folder / "video"

    for mp4 in src.glob("*.mp4"):
        video_id = mp4.stem

        rows.append({
            "kind": "video",
            "group": group,
            "video_id": video_id,
            "frame_name": None,
            "frame_idx": None,
            "source_path": str(mp4),
            "r2_key": f"Videos/Videos_{group}/{mp4.name}",
        })

df = pd.DataFrame(rows)
df.to_csv("/kaggle/working/media_manifest.csv", index=False)

print("Manifest rows:", len(df))
print(df["kind"].value_counts())
df.head()

In [ ]:
!aws --endpoint-url "$R2_ENDPOINT" s3 cp \
  /kaggle/working/media_manifest.csv \
  "s3://$R2_BUCKET/manifest/media_manifest.csv" \
  --content-type "text/csv" \
  --cache-control "public, max-age=3600"

In [ ]:
!aws --endpoint-url "$R2_ENDPOINT" s3 ls "s3://$R2_BUCKET/" 

In [ ]:
!aws --endpoint-url "$R2_ENDPOINT" s3 ls "s3://$R2_BUCKET/Keyframes/" 

In [ ]:
!aws --endpoint-url "$R2_ENDPOINT" s3 ls "s3://$R2_BUCKET/Videos/" 

In [ ]:
!aws --endpoint-url "$R2_ENDPOINT" s3 ls "s3://$R2_BUCKET/manifest/" 

In [ ]:
!aws --endpoint-url "$R2_ENDPOINT" s3 ls "s3://$R2_BUCKET/Videos/Videos_L25/" | head -20

In [ ]:
local_l25_a = None

for folder, group in video_folders:
    if folder.name == "Videos_L25_a":
        local_l25_a = folder / "video"

if local_l25_a:
    local_count = len(list(local_l25_a.glob("*.mp4")))
    print("Local Videos_L25_a count:", local_count)
else:
    print("Videos_L25_a not found in upload list")